# Whisper で日本語の音声を書き起こす（モデルの比較、長い音声、字幕）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/01_data_io/whisper_asr.ipynb)

NLP クックブック 01-06 ／ データの収集と入出力

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

録音した音声（講演、会議、インタビュー）を日本語のテキストに書き起こして、NLP の材料にしたい。どのモデルを使えばどれくらい正確で速いか、長い音声や字幕 (SRT) はどう扱うか、どんな誤りが出るかも確かめたい。

In [1]:
!pip install -q jiwer silero-vad

In [2]:
import io
import numpy as np
import pandas as pd
import soundfile as sf
from huggingface_hub import hf_hub_download
from IPython.display import Audio

# FLEURS（Google、CC BY 4.0）の日本語の開発用データ: 機械翻訳の評価データ FLoRes の文（英語の記事の翻訳）を読み上げた音声と、その文
# Common Voice の日本語なども使えるが、配布元の利用条件を確かめる
path = hf_hub_download("google/fleurs", "parquet-data/ja_jp/validation-00000-of-00001.parquet", repo_type="dataset")   # 約 220MB
df = pd.read_parquet(path).drop_duplicates("id").sort_values("id").reset_index(drop=True)   # 同じ文を複数の話者が読んでいるので 1 文 1 録音にする
SR = 16000                                                   # Whisper は 16kHz のモノラルを前提にする
df["audio"] = [sf.read(io.BytesIO(a["bytes"]), dtype="float32") for a in df["audio"]]
assert all(sr == SR for _, sr in df["audio"])                # ほかの音声は ffmpeg -ar 16000 -ac 1 で 16kHz のモノラルにしてから渡す
df["audio"] = [x for x, _ in df["audio"]]
print(len(df), "文", round(sum(map(len, df["audio"])) / SR / 60, 1), "分")
print(df["raw_transcription"][3])
Audio(df["audio"][3], rate=SR)

135 文 29.3 分
アルゼンチンは、世界最高のポロチームとポロ競技選手を擁することで有名です。


In [3]:
import time
import unicodedata
import jiwer
import torch
from transformers import pipeline

def normalize(s):
    """CER を測る前の正規化: 全角半角をそろえ、句読点・記号と空白を除く"""
    s = unicodedata.normalize("NFKC", s)
    return "".join(c for c in s if not unicodedata.category(c).startswith(("P", "Z")))

BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
DTYPE = torch.bfloat16 if BF16 else torch.float16
GEN = {"language": "ja", "task": "transcribe"}          # 言語を指定しないと、最初の 30 秒から言語を推定する
refs = [normalize(s) for s in df["raw_transcription"]]
audio_sec = sum(map(len, df["audio"])) / SR

MODELS = {"small": "openai/whisper-small",                       # Apache 2.0
          "large-v3-turbo": "openai/whisper-large-v3-turbo",     # MIT
          "kotoba-v2.0": "kotoba-tech/kotoba-whisper-v2.0"}      # Apache 2.0（日本語のテレビ音声 ReazonSpeech で蒸留）
hyps, rows = {}, []
for short, name in MODELS.items():                     # 3 モデルで T4 なら 5〜10 分程度（ダウンロードを含む）
    asr = pipeline("automatic-speech-recognition", model=name, dtype=DTYPE, device_map="auto")
    asr(df["audio"][0], generate_kwargs=GEN)           # 1 回目は初期化で遅いので、測る前に 1 回動かす
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t = time.time()
    out = asr(list(df["audio"]), batch_size=16, generate_kwargs=GEN)
    sec = time.time() - t
    hyps[short] = [o["text"] for o in out]
    rows.append({"モデル": short, "パラメータ（億）": asr.model.num_parameters() / 1e8,
                 "CER": jiwer.cer(refs, [normalize(h) for h in hyps[short]]), "時間（秒）": sec,
                 "実時間の何倍速": audio_sec / sec, "最大メモリ (GB)": torch.cuda.max_memory_allocated() / 1e9})
    del asr; torch.cuda.empty_cache()
pd.DataFrame(rows).round(3)

,モデル,パラメータ（億）,CER,時間（秒）,実時間の何倍速,最大メモリ (GB)
0,small,2.417,0.114,26.184,67.160,5.744
1,large-v3-turbo,8.089,0.038,21.174,83.051,5.044
2,kotoba-v2.0,7.564,0.054,13.979,125.798,3.707


In [4]:
# 文単位で再標本化するブートストラップで、CER の 95% 信頼区間とモデル間の差の信頼区間を求める
def errors(ref, hyp):
    m = jiwer.process_characters(ref, normalize(hyp))
    return m.substitutions + m.deletions + m.insertions

n_chars = np.array([len(r) for r in refs])
err = {k: np.array([errors(r, h) for r, h in zip(refs, v)]) for k, v in hyps.items()}
rng = np.random.default_rng(0)
idx = rng.integers(0, len(refs), (2000, len(refs)))
boot = {k: e[idx].sum(1) / n_chars[idx].sum(1) for k, e in err.items()}
for k, b in boot.items():
    print(f"{k:15s} CER {err[k].sum() / n_chars.sum():.3f}  95% CI [{np.percentile(b, 2.5):.3f}, {np.percentile(b, 97.5):.3f}]")
for a, b in [("small", "large-v3-turbo"), ("kotoba-v2.0", "large-v3-turbo")]:
    d = boot[a] - boot[b]
    print(f"{a} - {b}: {d.mean():+.3f}  95% CI [{np.percentile(d, 2.5):+.3f}, {np.percentile(d, 97.5):+.3f}]")

small           CER 0.114  95% CI [0.100, 0.129]
large-v3-turbo  CER 0.038  95% CI [0.029, 0.048]
kotoba-v2.0     CER 0.054  95% CI [0.043, 0.067]
small - large-v3-turbo: +0.076  95% CI [+0.064, +0.089]
kotoba-v2.0 - large-v3-turbo: +0.016  95% CI [+0.010, +0.022]


In [5]:
import re

# 典型的な誤り: 数字、カタカナ語（外国の固有名詞を含む）、漢字の語が、語の単位でどれくらい正しく出るか
def hit_rate(pattern, model):
    found = total = 0
    for r, h in zip(refs, hyps[model]):
        words = re.findall(pattern, r)
        found += sum(w in normalize(h) for w in words); total += len(words)
    return found / total, total

for model in hyps:
    rates = {k: hit_rate(p, model) for k, p in [("数字", r"\d+"), ("カタカナ 3 字以上", r"[ァ-ヴー]{3,}"), ("漢字 3 字以上", r"[一-龥]{3,}")]}
    print(f"{model:15s}", "  ".join(f"{k} {r:.2f}（{n} 個）" for k, (r, n) in rates.items()), f"  1 - CER {1 - err[model].sum() / n_chars.sum():.2f}")

# large-v3-turbo の誤りが多い文
for i in np.argsort(-err["large-v3-turbo"] / n_chars)[:6]:
    print(f"\n正解  : {df['raw_transcription'][i]}")
    for model in ["large-v3-turbo", "kotoba-v2.0"]:
        print(f"{model:6s}: {hyps[model][i]}")

small           数字 0.91（67 個）  カタカナ 3 字以上 0.72（196 個）  漢字 3 字以上 0.67（229 個）   1 - CER 0.89
large-v3-turbo  数字 0.96（67 個）  カタカナ 3 字以上 0.85（196 個）  漢字 3 字以上 0.90（229 個）   1 - CER 0.96
kotoba-v2.0     数字 0.93（67 個）  カタカナ 3 字以上 0.82（196 個）  漢字 3 字以上 0.84（229 個）   1 - CER 0.95

正解  : 両替は、スタンレーにある島内唯一の銀行（FIC Westの向かい）でできます。
large-v3-turbo: 業外はスタンレイにある東南唯一の銀行FICウエストの向かいでできます
kotoba-v2.0: 両替はスタンレーにある当な唯一の銀行FICUSとの向かいでできます

正解  : しかし、カタルーニャ語が第一公用語として法律で定められているため、ほとんどの標識がカタルーニャ語のみで表示されています。
large-v3-turbo: しかし語るうね語が第一公用語として法律で定められているためほとんどの標識が語るうね語のみで表示されています
kotoba-v2.0: しかしカタルウネ語が第一公営語として法律で定められているためほとんどの標識がカタルウネ語のみで表示されています

正解  : 新王国の古代エジプト人たちは、当時千年以上前に建てられた前身のモニュメントに驚嘆しました。
large-v3-turbo: 新王国の古代イジプト人たちは当時1000年以上前に建てられた全身のモニュメントに共談しました
kotoba-v2.0: 新王国の古代イジブト人たちは当時1000年以上前に建てられた前身のモリュメントに驚嘆しました

正解  : 例えば、インターネット利用の重要な動機として「学習」や「社会化」が示唆されています(James et al., 1995)。
large-v3-turbo: 例えばインターネット利用の重要な動機として、学習や社会化が示唆されています。ジェームス・エト・アル、1995
kotoba-v2.0: 例えばインターネット利用の重要な動機でして学習や社会科がしさされていますジェームスエットR

In [6]:
# 長い音声: 15 文をつなげた約 4 分の音声（文の間に 1 秒の無音）。録音ごとに音量が違うのは、会議で話者とマイクの距離が違うのに近い
long_audio = np.concatenate([x for a in df["audio"][:15] for x in (a, np.zeros(SR, np.float32))])
long_ref = normalize("".join(df["raw_transcription"][:15]))
sf.write("long.wav", long_audio, SR)
# ffmpeg の dynaudnorm で、短い区間ごとに音量をそろえる（f: 区間の長さ（ミリ秒）、g: 平滑化する区間数、m: 最大の増幅率）
!ffmpeg -y -loglevel error -i long.wav -af dynaudnorm=f=200:g=11:m=100 long_norm.wav
long_norm = sf.read("long_norm.wav", dtype="float32")[0]

starts = np.cumsum([0] + [len(a) + SR for a in df["audio"][:15]])
db = lambda x: 20 * np.log10(np.sqrt(np.mean(x ** 2)) + 1e-9)
print(round(len(long_audio) / SR), "秒")
print("文ごとの音量 (dBFS)       :", [round(db(long_audio[s:e - SR])) for s, e in zip(starts, starts[1:])])
print("文ごとの音量（そろえたあと）:", [round(db(long_norm[s:e - SR])) for s, e in zip(starts, starts[1:])])

231 秒
文ごとの音量 (dBFS)       : [-17, -61, -19, -18, -25, -22, -19, -20, -25, -25, -24, -60, -22, -18, -61]
文ごとの音量（そろえたあと）: [-17, -22, -17, -18, -16, -19, -18, -18, -16, -15, -16, -22, -17, -16, -23]


In [7]:
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo", dtype=DTYPE, device_map="auto")

def report(name, text, sec):
    m = jiwer.process_characters(long_ref, normalize(text))
    print(f"{name:30s} CER {m.cer:.3f}（置換 {m.substitutions:3d} / 削除 {m.deletions:3d} / 挿入 {m.insertions:3d}）、{sec:.1f} 秒")

# 30 秒を超える音声は、chunk_length_s で区切るか、return_timestamps=True で逐次処理する（どちらもないと ValueError になる）
results = {}
for name, audio, kw in [("逐次処理（音量そのまま）", long_audio, dict(return_timestamps=True)),
                        ("逐次処理", long_norm, dict(return_timestamps=True)),
                        ("chunk_length_s=30", long_norm, dict(chunk_length_s=30, batch_size=8)),
                        ("chunk_length_s=30 + タイムスタンプ", long_norm, dict(chunk_length_s=30, batch_size=8, return_timestamps=True))]:
    t = time.time()
    results[name] = asr(audio, generate_kwargs=GEN, **kw)
    report(name, results[name]["text"], time.time() - t)

# 文ごとに、正解の文字のうち認識結果に正しく出た割合を見る
bounds = np.cumsum([0] + [len(normalize(t)) for t in df["raw_transcription"][:15]])
def sentence_recall(text):
    ok = np.zeros(len(long_ref), bool)
    for c in jiwer.process_characters(long_ref, normalize(text)).alignments[0]:
        if c.type == "equal":
            ok[c.ref_start_idx:c.ref_end_idx] = True
    return [round(float(ok[a:b].mean()), 2) for a, b in zip(bounds, bounds[1:])]

for name in ["逐次処理（音量そのまま）", "逐次処理", "chunk_length_s=30"]:
    print(f"{name:24s}", sentence_recall(results[name]["text"]))
print(results["chunk_length_s=30"]["text"][:150])

# タイムスタンプ付きの分割で、正解より多く出てくる 12 文字の並び（二重に出たか、挿入されたところ）
h = normalize(results["chunk_length_s=30 + タイムスタンプ"]["text"])
dups = sorted({h[i:i + 12] for i in range(len(h) - 11) if h.count(h[i:i + 12]) > long_ref.count(h[i:i + 12])})
print(len(dups), "種類:", dups[:4])

逐次処理（音量そのまま）                   CER 0.136（置換  18 / 削除  87 / 挿入   4）、4.8 秒
逐次処理                           CER 0.019（置換  11 / 削除   1 / 挿入   3）、5.0 秒
chunk_length_s=30              CER 0.684（置換 393 / 削除  28 / 挿入 128）、6.6 秒
chunk_length_s=30 + タイムスタンプ    CER 0.187（置換  14 / 削除   2 / 挿入 134）、3.7 秒
逐次処理（音量そのまま）             [0.92, 0.0, 1.0, 1.0, 1.0, 1.0, 0.95, 0.98, 0.96, 0.97, 0.97, 0.0, 1.0, 0.98, 0.89]
逐次処理                     [0.92, 1.0, 1.0, 1.0, 1.0, 1.0, 0.97, 0.98, 0.96, 0.97, 0.99, 1.0, 1.0, 1.0, 1.0]
chunk_length_s=30        [0.86, 0.0, 0.0, 0.2, 0.0, 0.04, 0.02, 0.05, 0.15, 0.4, 0.99, 1.0, 0.79, 0.89, 1.0]
また国連では地球温暖化の影響を被っているクリグリがその影響に対処できるように支援するための危機について最終決定しただったものとは異なる特徴や適応が必要となり、強力な競争相手となる可能性があります。アルゼンチンアルゼンチンアルゼンチンアルゼンチンアルゼンチンアルゼンチンアルゼンチンアルゼンチンアルゼン
268 種類: ['1818世紀の市場古いモ', '818世紀の市場古いモス', 'ありがとうございましたそ', 'ありがとうございました冬']


In [8]:
from silero_vad import load_silero_vad, get_speech_timestamps

# 無音のところで区切り、30 秒未満の区間をまとめてバッチで認識する
vad = load_silero_vad()                                # Silero VAD (MIT): 発話区間を検出するモデル
speech = get_speech_timestamps(torch.from_numpy(long_norm), vad, sampling_rate=SR, max_speech_duration_s=25)
segments = []                                          # 発話区間を、25 秒を超えない範囲でまとめる
for s in speech:
    if segments and s["end"] - segments[-1][0] <= 25 * SR:
        segments[-1][1] = s["end"]
    else:
        segments.append([s["start"], s["end"]])
print(len(speech), "個の発話区間 →", len(segments), "個の区間:", [round((e - s) / SR, 1) for s, e in segments])

t = time.time()
outs = asr([long_norm[s:e] for s, e in segments], batch_size=8, generate_kwargs=GEN, return_timestamps=True)
report("無音で区切ってバッチで認識", "".join(o["text"] for o in outs), time.time() - t)

34 個の発話区間 → 10 個の区間: [21.2, 24.1, 18.5, 22.6, 21.6, 24.3, 22.9, 3.2, 16.0, 8.3]
無音で区切ってバッチで認識                  CER 0.022（置換  14 / 削除   1 / 挿入   3）、2.6 秒


In [9]:
# タイムスタンプ付きの出力を字幕 (SRT) にする
def srt_time(t):
    ms = round(t * 1000)
    h, ms = divmod(ms, 3_600_000); m, ms = divmod(ms, 60_000); s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

subs = []
for (s, e), o in zip(segments, outs):
    for c in o["chunks"]:
        start, end = c["timestamp"]
        end = end if end is not None else (e - s) / SR          # 区間の最後で終わりの時刻が出ないことがある
        subs.append((s / SR + start, s / SR + end, c["text"].strip()))
srt = "\n".join(f"{i}\n{srt_time(a)} --> {srt_time(b)}\n{text}\n" for i, (a, b, text) in enumerate(subs, 1))
with open("long.srt", "w", encoding="utf-8") as f:
    f.write(srt)
print(len(subs), "行の字幕")
print(srt[:500])

24 行の字幕
1
00:00:02,210 --> 00:00:11,310
また国連では地球温暖化の影響をこむっているクリグリがその影響に対処できるように支援するための危機について最終決定したいと考えています。

2
00:00:16,690 --> 00:00:23,430
まもなく暴動・鎮圧用の装備を着用した警官たちが収容所に入り、

3
00:00:24,066 --> 00:00:27,146
催涙ガスで受刑者を追い詰めました

4
00:00:27,146 --> 00:00:39,226
この新しい環境には異なる資源と異なる競争相手が存在するため

5
00:00:39,226 --> 00:00:43,226
新しい集団にはそれまで必要だったものとは異なる特徴や

6
00:00:43,226 --> 00:00:47,926
適応が必要となり強力な競争相手となる可能性があります

7
00:00:54,658 --> 00:00:56,898
アルゼンチンは世界最高のプロチームと

8
00:00:56,898 --> 00:00:59,258
プロ競技選手を擁することで有名です。


In [10]:
# 幻覚と繰り返し: 音声のないところに文字を出す
rng = np.random.default_rng(0)
tests = {"無音 10 秒": np.zeros(10 * SR, np.float32),
         "雑音 10 秒": (rng.standard_normal(10 * SR) * 0.01).astype(np.float32),
         "発話のあとに無音 15 秒": np.concatenate([df["audio"][0], np.zeros(15 * SR, np.float32)])}
for name, x in tests.items():
    text = asr(x, generate_kwargs=GEN)["text"]
    n_speech = len(get_speech_timestamps(torch.from_numpy(x), vad, sampling_rate=SR))
    print(f"{name}: VAD の発話区間 {n_speech} 個 / 認識結果 {len(text)} 文字: {text if len(text) <= 100 else text[:50] + '…'}")
print("正解:", df["raw_transcription"][0])

無音 10 秒: VAD の発話区間 0 個 / 認識結果 14 文字: ご視聴ありがとうございました
雑音 10 秒: VAD の発話区間 0 個 / 認識結果 444 文字: 音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声音声…
発話のあとに無音 15 秒: VAD の発話区間 1 個 / 認識結果 65 文字: また国連では地球温暖化の影響を被っているクリグリがその影響に対処できるように支援するための危機について最終決定したいと考えています
正解: また、国連では、地球温暖化の影響を被っている国々がその影響に対処できるように支援するための基金について最終決定したいと考えています。


- FLEURS の 135 文（約 29 分）で、CER は large-v3-turbo 0.038、kotoba-whisper-v2.0 0.054、small 0.114 で、差はブートストラップの信頼区間で 0 を含まない。FLEURS は英語の記事を訳した文の読み上げで外国の固有名詞が多く、話し言葉では順位が変わりうる（kotoba のモデルカードでは ReazonSpeech の評価データで large-v3 より CER が低い）。
- 検証に使った GPU（bf16 で実行）では、kotoba は turbo より速く（14 秒と 21 秒）、small は小さいのに速くなかった（26 秒）。1 トークンずつ生成するデコーダの層数（small 12、turbo 4、kotoba 2）が効くためと考えられる。GPU メモリの最大は 5.7GB（small、バッチ 16）。T4 では float16 で動くので、時間とメモリはこれと違う。
- 語の単位で正しく出た割合は、3 字以上のカタカナ語（turbo 0.85、kotoba 0.82、small 0.72）も漢字の語（0.90、0.84、0.67）も 1 − CER より低い。1 字違えば外れになるためで、カタカナ語（外国の固有名詞を含む）だけに誤りが集中するとは言えない。「カタルーニャ語 → 語るうね語」「両替 → 業外」「千年 → 1000年」のような誤りや表記の違いが出る（正規化は評価指標と統計的検定の章の「音声認識や OCR の誤り率 (WER / CER) を計算する」を参照）。
- 約 4 分の音声 1 本で、逐次処理（`return_timestamps=True`）は CER 0.019 だったが、`chunk_length_s=30` で分けると繰り返しや文の脱落で 0.684、タイムスタンプを付けても二重の出力や挿入で 0.187 になった。モデルカードも精度を優先するなら逐次処理、速さなら分割としているが、この例では分割による精度の落ち方が大きい。
- 音量が 40dB ほど小さい 3 文のうち 2 文は丸ごと抜けた。ffmpeg の dynaudnorm で音量をそろえると、どの文も 9 割以上の文字が出た。
- Silero VAD で無音のところで区切ってバッチで認識すると、CER 0.022 と逐次処理に近く、時間は短かった（逐次処理の 5.0 秒に対して 2.6 秒。検証に使った GPU での値）。区間の開始時刻を足せば字幕 (SRT) になる。句読点が要るなら日本語の形態素解析と文分割の章の「句読点のない文章に読点と句点を付ける（句読点の復元）」を参照。
- 無音や雑音にも文字を出す（幻覚）が、VAD はどちらも発話区間 0 個と判定したので、VAD で区切れば防げる。発話のあとの無音では、この例では幻覚は出なかった。
- 書き起こしの先は、固有表現抽出の章の「音声認識の結果から固有表現を抽出する」と、要約と平易化の章の「会議の書き起こしから議事録を作る（決定事項・ToDo を JSON で抜き出す）」を参照。議事録には pyannote などの話者分離を組み合わせる（モデルの利用条件への同意と HF のトークンが要る。kotoba-whisper-v2.2 は話者分離を組み込んだパイプライン）。字幕を作って翻訳する例は、機械翻訳の章の「動画の字幕を作って翻訳する（区切り直し、文脈、文字数の制約）」を参照。